In [1]:
pip install langchain langchain-openai langchain-community sqlalchemy pandas openai

   ---------------------------------------- 0.0/548.1 kB ? eta -:--:--
   ---------------------------------------- 548.1/548.1 kB 13.1 MB/s  0:00:00
   ---------------------------------------- 0.0/874.8 kB ? eta -:--:--
   ---------------------------------------- 874.8/874.8 kB 23.4 MB/s  0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ----------------- ---------------------- 1.0/2.4 MB 9.2 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 5.9 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ------------------------------ --------- 0.8/1.0 MB 35.5 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 9.1 MB/s  0:00:00

   -------- -------------------------------  4/18 [langchain-protocol]
   ------------- --------------------------  6/18 [tiktoken]
   --------------- ------------------------  7/18 [langsmith]
   --------------- ------------------------  7/18 [langsmith]
   -----

In [2]:
import pandas as pd
from sqlalchemy import create_engine
import numpy as np

# Create a local SQLite database (no server needed!)
engine = create_engine('sqlite:///retail.db')

# --- Sales Fact Table ---
np.random.seed(42)
categories = ['Patio', 'Footwear', 'Tools', 'Apparel', 'BBQ']
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']
stores = ['Toronto', 'Brampton', 'Mississauga', 'Scarborough']

rows = []
for cat in categories:
    for month in months:
        for store in stores:
            sales = np.random.randint(10000, 100000)
            units = np.random.randint(50, 500)
            aur = round(sales / units, 2)
            yoy = round(np.random.uniform(-30, 30), 2)
            rows.append([cat, month, store, sales, units, aur, yoy])

sales_df = pd.DataFrame(rows, columns=[
    'category', 'month', 'store',
    'sales_amount', 'units_sold',
    'aur', 'yoy_change'
])

# --- SKU Master Table ---
skus = []
for i in range(20):
    skus.append([
        f'SKU{i:03d}',
        f'Product {i}',
        np.random.choice(categories),
        np.random.choice(['Premium', 'Mid', 'Budget']),
        round(np.random.uniform(10, 500), 2)
    ])

sku_df = pd.DataFrame(skus, columns=[
    'sku_id', 'sku_name',
    'category', 'product_tier', 'aur'
])

# Save to database
sales_df.to_sql('sales_fact', engine, if_exists='replace', index=False)
sku_df.to_sql('sku_master', engine, if_exists='replace', index=False)

print("✅ Database created successfully!")
print(f"Sales records: {len(sales_df)}")
print(f"SKU records: {len(sku_df)}")

✅ Database created successfully!
Sales records: 120
SKU records: 20


In [4]:
import os
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain_openai import ChatOpenAI

# Connect to your database
db = SQLDatabase(engine)

# GPT-4o as the brain
llm = ChatOpenAI(
    model="gpt-4o",
    api_key="your_api_key",  # paste your key here
    temperature=0
)

# Create the autonomous agent
agent = create_sql_agent(
    llm=llm,
    db=db,
    verbose=True,  # shows its thinking step by step
    agent_type="openai-tools"
)

print("✅ Agent ready!")

✅ Agent ready!


In [5]:
result = agent.run("""
You are a senior retail data analyst at a large Canadian retailer.

Analyze the sales data available to you and autonomously identify:
1. Top 3 performing categories by total sales
2. Bottom 3 declining categories by YoY change
3. Any anomalies or unexpected patterns you find
4. One actionable recommendation

For every insight show me the exact numbers you found.
Be specific. Sound like a human analyst.
""")

print(result)

C:\Users\twink\AppData\Local\Temp\ipykernel_7792\1564909402.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = agent.run("""




> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


sales_fact, sku_master
Invoking: `sql_db_schema` with `{'table_names': 'sales_fact, sku_master'}`



CREATE TABLE sales_fact (
	category TEXT, 
	month TEXT, 
	store TEXT, 
	sales_amount BIGINT, 
	units_sold BIGINT, 
	aur FLOAT, 
	yoy_change FLOAT
)

/*
3 rows from sales_fact table:
category	month	store	sales_amount	units_sold	aur	yoy_change
Patio	Jan	Toronto	25795	398	64.81	-18.99
Patio	Jan	Brampton	86820	152	571.18	-3.25
Patio	Jan	Mississauga	47194	137	344.48	-9.98
*/


CREATE TABLE sku_master (
	sku_id TEXT, 
	sku_name TEXT, 
	category TEXT, 
	product_tier TEXT, 
	aur FLOAT
)

/*
3 rows from sku_master table:
sku_id	sku_name	category	product_tier	aur
SKU000	Product 0	Tools	Budget	23.53
SKU001	Product 1	Footwear	Mid	224.85
SKU002	Product 2	Tools	Premium	85.97
*/
Invoking: `sql_db_query_checker` with `{'query': 'SELECT category, SUM(sales_amount) AS total_sales FROM sales_fact GROUP BY category ORD

In [6]:
result2 = agent.run("""
Dig deeper into the Footwear category.
Which stores are driving the YoY decline?
Are there any stores bucking the trend?
Show me exact numbers.
""")
print(result2)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


sales_fact, sku_master
Invoking: `sql_db_schema` with `{'table_names': 'sales_fact, sku_master'}`



CREATE TABLE sales_fact (
	category TEXT, 
	month TEXT, 
	store TEXT, 
	sales_amount BIGINT, 
	units_sold BIGINT, 
	aur FLOAT, 
	yoy_change FLOAT
)

/*
3 rows from sales_fact table:
category	month	store	sales_amount	units_sold	aur	yoy_change
Patio	Jan	Toronto	25795	398	64.81	-18.99
Patio	Jan	Brampton	86820	152	571.18	-3.25
Patio	Jan	Mississauga	47194	137	344.48	-9.98
*/


CREATE TABLE sku_master (
	sku_id TEXT, 
	sku_name TEXT, 
	category TEXT, 
	product_tier TEXT, 
	aur FLOAT
)

/*
3 rows from sku_master table:
sku_id	sku_name	category	product_tier	aur
SKU000	Product 0	Tools	Budget	23.53
SKU001	Product 1	Footwear	Mid	224.85
SKU002	Product 2	Tools	Premium	85.97
*/
Invoking: `sql_db_query_checker` with `{'query': "SELECT store, SUM(yoy_change) AS total_yoy_change FROM sales_fact WHERE category = 'Foo

In [7]:
result2 = agent.run("""
Dig deeper into the Footwear category.
Which stores are driving the YoY decline?
Are there any stores bucking the trend and showing positive YoY?
What is the AUR trend across stores?
Show me exact numbers for everything.
""")
print(result2)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


sales_fact, sku_master
Invoking: `sql_db_schema` with `{'table_names': 'sales_fact, sku_master'}`



CREATE TABLE sales_fact (
	category TEXT, 
	month TEXT, 
	store TEXT, 
	sales_amount BIGINT, 
	units_sold BIGINT, 
	aur FLOAT, 
	yoy_change FLOAT
)

/*
3 rows from sales_fact table:
category	month	store	sales_amount	units_sold	aur	yoy_change
Patio	Jan	Toronto	25795	398	64.81	-18.99
Patio	Jan	Brampton	86820	152	571.18	-3.25
Patio	Jan	Mississauga	47194	137	344.48	-9.98
*/


CREATE TABLE sku_master (
	sku_id TEXT, 
	sku_name TEXT, 
	category TEXT, 
	product_tier TEXT, 
	aur FLOAT
)

/*
3 rows from sku_master table:
sku_id	sku_name	category	product_tier	aur
SKU000	Product 0	Tools	Budget	23.53
SKU001	Product 1	Footwear	Mid	224.85
SKU002	Product 2	Tools	Premium	85.97
*/
Invoking: `sql_db_query_checker` with `{'query': "SELECT store, SUM(yoy_change) AS total_yoy_change FROM sales_fact WHERE category = 'Foo

In [8]:
result3 = agent.run("""
Based on everything you know about the Footwear category 
across all stores, write a 3 paragraph executive commentary 
that a senior retail analyst would present to leadership.

Include:
- Overall category health with specific numbers
- Which stores are concerning and why
- Which store is a positive outlier and what we can learn from it
- One clear recommendation

Write it in professional business language. 
No bullet points — flowing paragraphs only.
""")

print(result3)



> Entering new SQL Agent Executor chain...

Invoking: `sql_db_list_tables` with `{}`


sales_fact, sku_master
Invoking: `sql_db_schema` with `{'table_names': 'sales_fact, sku_master'}`



CREATE TABLE sales_fact (
	category TEXT, 
	month TEXT, 
	store TEXT, 
	sales_amount BIGINT, 
	units_sold BIGINT, 
	aur FLOAT, 
	yoy_change FLOAT
)

/*
3 rows from sales_fact table:
category	month	store	sales_amount	units_sold	aur	yoy_change
Patio	Jan	Toronto	25795	398	64.81	-18.99
Patio	Jan	Brampton	86820	152	571.18	-3.25
Patio	Jan	Mississauga	47194	137	344.48	-9.98
*/


CREATE TABLE sku_master (
	sku_id TEXT, 
	sku_name TEXT, 
	category TEXT, 
	product_tier TEXT, 
	aur FLOAT
)

/*
3 rows from sku_master table:
sku_id	sku_name	category	product_tier	aur
SKU000	Product 0	Tools	Budget	23.53
SKU001	Product 1	Footwear	Mid	224.85
SKU002	Product 2	Tools	Premium	85.97
*/
Invoking: `sql_db_query_checker` with `{'query': "SELECT store, SUM(sales_amount) AS total_sales, SUM(units_sold) AS total_units, AVG(aur